In [ ]:
%pip install pretty_midi torch numpy matplotlib scikit-learn

## 1. Data Loading
Load the raw MIDI files from the project dataset and gather all file paths used for preprocessing.

In [23]:
from pathlib import Path
import pretty_midi
import numpy as np

PROJECT_ROOT = Path("..")      #
DATA_ROOT = PROJECT_ROOT / "data" / "classic-midi-files-raw"
print("DATA_ROOT:", DATA_ROOT.resolve(), "exists:", DATA_ROOT.exists())

COMPOSERS = ["Beethoven", "Bach"]

midi_files = []
for comp in COMPOSERS:
    comp_dir = DATA_ROOT / comp
    midi_files.extend(sorted(comp_dir.rglob("*.mid")))

print(f"Found {len(midi_files)} MIDI files total from {COMPOSERS}.")
print("First few files:")
for p in midi_files[:5]:
    print(" ", p)
                      

DATA_ROOT: /Users/catherinembata/music-generation-project/data/classic-midi-files-raw exists: True
Found 1137 MIDI files total from ['Beethoven', 'Bach'].
First few files:
  ../data/classic-midi-files-raw/Beethoven/32 Variations on a theme.mid
  ../data/classic-midi-files-raw/Beethoven/Andante in F Major.mid
  ../data/classic-midi-files-raw/Beethoven/Anh06 Rondo.mid
  ../data/classic-midi-files-raw/Beethoven/Anh08Nb1 Gavotte 4 hands.mid
  ../data/classic-midi-files-raw/Beethoven/Anhang 14-3.mid


## 2. Preprocessing & Tokenization
Extract the melody from each MIDI file, convert individual notes into (pitch, duration bucket) tokens, and handle data-quality issues by skipping unusable or too-short files

In [24]:
def choose_melody_instrument(midi: pretty_midi.PrettyMIDI):
    """Return a single pretty_midi.Instrument to treat as the melody, or None."""
    # Filter out drums
    candidates = [inst for inst in midi.instruments if not inst.is_drum]
    if not candidates:
        return None
    
    # Prefer Acoustic Grand Piano (program 0), but fall back to all non-drum
    piano_candidates = [inst for inst in candidates if inst.program == 0]
    if piano_candidates:
        candidates = piano_candidates
    
    # If instrument has no notes, skip it
    candidates = [inst for inst in candidates if inst.notes]
    if not candidates:
        return None
    
    # Pick instrument with highest average pitch (melody heuristic)
    def avg_pitch(inst):
        return np.mean([n.pitch for n in inst.notes])
    
    melody_inst = max(candidates, key=avg_pitch)
    return melody_inst


def duration_to_bucket(duration: float) -> int:
    """
    Map a duration in seconds to a small number of buckets.
    You can tweak these thresholds later.
    """
    if duration < 0.2:
        return 0  # very short (eighth-ish)
    elif duration < 0.5:
        return 1  # short (quarter-ish)
    elif duration < 1.0:
        return 2  # medium (half-ish)
    else:
        return 3  # long (whole+)


def notes_to_tokens(notes):
    """
    Convert a list of pretty_midi.Note objects into (pitch, duration_bucket) tokens,
    sorted by start time.
    """
    # Sort notes by start time
    notes = sorted(notes, key=lambda n: n.start)
    
    tokens = []
    for note in notes:
        pitch = note.pitch
        duration = note.end - note.start
        bucket = duration_to_bucket(duration)
        tokens.append((pitch, bucket))
    return tokens


In [25]:
# Build token sequences from ALL discovered MIDI files

SEQ_LEN = 50  # context window length

all_token_seqs = []      # list of lists of (pitch, bucket)
seq_file_paths = []      # parallel list of file paths (as strings)

num_loaded = 0
num_skipped_too_short = 0
num_skipped_no_melody = 0
num_failed_load = 0

for midi_path in midi_files:
    try:
        midi = pretty_midi.PrettyMIDI(str(midi_path))
    except Exception:
        num_failed_load += 1
        continue

    melody = choose_melody_instrument(midi)
    if melody is None:
        num_skipped_no_melody += 1
        continue

    tokens = notes_to_tokens(melody.notes)

    if len(tokens) < SEQ_LEN + 1:
        num_skipped_too_short += 1
        continue

    all_token_seqs.append(tokens)
    seq_file_paths.append(str(midi_path))
    num_loaded += 1


# Summary
print("\n=== Summary ===")
print("Total files found:        ", len(midi_files))
print("Successfully loaded:      ", num_loaded)
print("Skipped (no melody):      ", num_skipped_no_melody)
print("Skipped (too short):      ", num_skipped_too_short)
print("Failed to load:           ", num_failed_load)
print("Usable token sequences:   ", len(all_token_seqs))



=== Summary ===
Total files found:         1137
Successfully loaded:       979
Skipped (no melody):       0
Skipped (too short):       157
Failed to load:            1
Usable token sequences:    979


## 3. Dataset construction (X, y)
This section converts token sequences into numerical IDs, builds the vocabulary, and constructs input sequences (X) and next-token targets (y) for training.

In [26]:
# Build vocabulary from all (pitch, duration_bucket) tokens

token_to_id = {}
for seq in all_token_seqs:
    for tok in seq:
        if tok not in token_to_id:
            token_to_id[tok] = len(token_to_id)

id_to_token = {i: tok for tok, i in token_to_id.items()}

print("Vocab size:", len(token_to_id))

Vocab size: 307


In [27]:
# Build X (contexts), y (next-token targets), and example_file_indices

X_ids = []
y_ids = []
example_file_indices = []  # index into seq_file_paths

for file_idx, seq in enumerate(all_token_seqs):
    # Convert (pitch, bucket) tokens to IDs
    seq_ids = [token_to_id[t] for t in seq]

    if len(seq_ids) <= SEQ_LEN:
        continue  # already filtered earlier, but just in case

    # Slide a window of length SEQ_LEN
    for i in range(len(seq_ids) - SEQ_LEN):
        context = seq_ids[i : i + SEQ_LEN]
        target = seq_ids[i + SEQ_LEN]

        X_ids.append(context)
        y_ids.append(target)
        example_file_indices.append(file_idx)

X_ids = np.array(X_ids, dtype=np.int64)
y_ids = np.array(y_ids, dtype=np.int64)
example_file_indices = np.array(example_file_indices, dtype=np.int64)

print("X shape:", X_ids.shape)
print("y shape:", y_ids.shape)
print("Number of examples:", len(X_ids))
print("Unique source files used:", len(set(example_file_indices)))


X shape: (683118, 50)
y shape: (683118,)
Number of examples: 683118
Unique source files used: 979


In [28]:
from pathlib import Path
import numpy as np

out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "full_sequences.npz"

vocab_array = np.array(list(token_to_id.keys()), dtype=np.int64)

np.savez(
    out_path,
    X=X_ids,
    y=y_ids,
    file_indices=example_file_indices,
    vocab=vocab_array,
    seq_len=SEQ_LEN,
)

print("Saved full dataset to:", out_path)


Saved full dataset to: data/processed/full_sequences.npz


## 4. Train/Val/Test Split
I perform a file-level split to avoid data leakage, creating separate train/validation/test sets.

In [29]:
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)

unique_files = np.unique(example_file_indices)
n_files = len(unique_files)
print("Total unique source files:", n_files)

# First split: train vs (val+test) – 70% / 30%
train_files, temp_files = train_test_split(
    unique_files,
    test_size=0.30,
    random_state=42,
    shuffle=True,
)

# Second split: val vs test – split temp 50/50 → 15% / 15% overall
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

print("Train files:", len(train_files))
print("Val files:  ", len(val_files))
print("Test files: ", len(test_files))

# Build boolean masks over examples
train_mask = np.isin(example_file_indices, train_files)
val_mask   = np.isin(example_file_indices, val_files)
test_mask  = np.isin(example_file_indices, test_files)

# Convert masks to index arrays (positions in X_ids / y_ids)
train_idx = np.where(train_mask)[0]
val_idx   = np.where(val_mask)[0]
test_idx  = np.where(test_mask)[0]

print("\nExamples per split:")
print("  Train examples:", len(train_idx))
print("  Val examples:  ", len(val_idx))
print("  Test examples: ", len(test_idx))

total = len(train_idx) + len(val_idx) + len(test_idx)
print("\nTotal examples accounted for:", total, " (expected:", len(X_ids), ")")


Total unique source files: 979
Train files: 685
Val files:   147
Test files:  147

Examples per split:
  Train examples: 487526
  Val examples:   92563
  Test examples:  103029

Total examples accounted for: 683118  (expected: 683118 )


## 5. Baseline Model (Most Frequent Token)
Compute the accuracy of a very simple model that always predicts the most common next-token in the training set. This establishes a lower bound to compare the GRU against.

In [ ]:
# Majority-token baseline: always predict the most frequent y in the TRAIN set

# Get train/val/test target arrays
y_train = y_ids[train_idx]
y_val   = y_ids[val_idx]
y_test  = y_ids[test_idx]

# Find the most common token in training labels
(unique, counts) = np.unique(y_train, return_counts=True)
majority_token_id = unique[np.argmax(counts)]
majority_count = counts.max()

print("Majority token ID:", majority_token_id)
print("Appears in train labels:", majority_count, "times")

def majority_baseline_accuracy(y, majority_id):
    return np.mean(y == majority_id)

baseline_train_acc = majority_baseline_accuracy(y_train, majority_token_id)
baseline_val_acc   = majority_baseline_accuracy(y_val, majority_token_id)
baseline_test_acc  = majority_baseline_accuracy(y_test, majority_token_id)

print("\nMajority-token baseline accuracy:")
print(f"  Train: {baseline_train_acc:.4f}")
print(f"  Val:   {baseline_val_acc:.4f}")
print(f"  Test:  {baseline_test_acc:.4f}")

## 6. Save Processed Dataset
Save X, y, vocabulary, and split indices into compressed `.npz` files for use in training and generation notebooks.

In [30]:
from pathlib import Path
import numpy as np

# Directory where you already saved full_sequences.npz
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

splits_path = out_dir / "splits_filelevel_indices.npz"

np.savez(
    splits_path,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    train_files=train_files,
    val_files=val_files,
    test_files=test_files,
)

print("Saved split indices to:", splits_path)
print("  Train examples:", len(train_idx))
print("  Val examples:  ", len(val_idx))
print("  Test examples: ", len(test_idx))


Saved split indices to: data/processed/splits_filelevel_indices.npz
  Train examples: 487526
  Val examples:   92563
  Test examples:  103029
